# Data Manipulation — Advanced
## Feature Engineering (R / tidymodels)

**Philosophy:** This note starts where cleaning and reshaping end — data is
clean and in the right shape. Each section covers how to create new features
for ML/modeling, with explicit attention to *when to use each technique* and
*what can go wrong (especially leakage)*.

**A structural note on this port:** the Python source leans heavily on
scikit-learn's `Pipeline` and `ColumnTransformer` — objects that bundle a
sequence of column-wise transforms into one fit-on-train-only unit. R's
**tidymodels** ecosystem solves the identical problem with a different
object model: a single `recipes::recipe()` accumulates `step_*()` calls (one
per transformation), and `prep()`/`bake()` (or `juice()`) replace
`fit()`/`transform()`. There is no direct `ColumnTransformer` equivalent to
transliterate line-by-line — instead, **each `step_*()` call already targets
specific columns via `tidyselect` semantics** (`all_numeric()`,
`all_nominal()`, explicit names), which is what `ColumnTransformer` exists to
bolt onto sklearn's column-blind transformers. This section is therefore a
genuine redesign, not a mechanical translation — see §8 for the full
`recipe()` capstone, the direct structural analog to this note's closing
`ColumnTransformer` example.

---

## Decision Table

| If your column is... | Go to |
| :--- | :--- |
| Categorical / string that needs to be numeric | §1 — Encoding Categoricals |
| Numeric but skewed, on different scales, or needs normalization | §2 — Numerical Transformations |
| Continuous numeric that should be grouped into ranges | §3 — Binning & Discretization |
| A datetime that should produce predictive features | §4 — Date & Time Features |
| A time-ordered numeric where past values predict future | §5 — Lag & Window Features |
| Two or more columns whose combination may be informative | §6 — Interaction & Ratio Features |
| Free text in a tabular dataset | §7 — Text as Features |
| Suspiciously good model performance or target in features | §8 — Leakage & Validation |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` | Cleaning & QC — audit, types, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` | Reshaping & combining — wide/long, group-by, multi-table joins, time-series, nested, many files |
| `DM_Advanced` **(this note)** | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Tidyverse / Reference_BaseR` | Syntax lookups for individual functions |

> **Environment:** R 4.x / tidyverse 2.x / tidymodels 1.x. **Convention:**
> `X_train`/`X_test` are feature frames and `y_train` the target; in
> tidymodels idiom this is more often a single `train`/`test` split from
> `rsample`, with the target column kept alongside the features inside the
> `recipe()` — fit (`prep()`) every recipe on train only, then `bake()` test.
> This notebook's cells are reference-style (they assume your modelling
> data); the added cells below are self-contained and runnable.

---
## §1 — Encoding Categorical Variables

Most models require numeric input. The right encoding depends on cardinality,
whether order exists, and whether you're using a tree-based or linear model.

In [ ]:
library(dplyr)
library(recipes)

# Sample data for encoding examples
set.seed(42)
df <- tibble::tibble(
  platform   = sample(c("iOS","Android","Web"), 200, replace=TRUE),
  risk_level = sample(c("Low","Medium","High","Very High"), 200, replace=TRUE),
  city       = sample(paste0("City", 1:20), 200, replace=TRUE),
  spend      = round(runif(200, 10, 500), 2),
  churn      = rbinom(200, 1, 0.3)
)
df_train <- df[1:150, ]; df_test <- df[151:200, ]

# ── Encoding 1: One-Hot Encoding via recipes ──────────────────────────────────
rec_ohe <- recipe(~ platform, data = df_train) |>
  step_dummy(platform, one_hot = TRUE) |>
  prep(training = df_train)

train_ohe <- bake(rec_ohe, new_data = NULL)
test_ohe  <- bake(rec_ohe, new_data = df_test)
cat("One-hot encoded columns:", names(train_ohe), "\n")
cat("Train rows:", nrow(train_ohe), "  Test rows:", nrow(test_ohe), "\n")

# ── Encoding 2: Ordinal encoding ──────────────────────────────────────────────
order_map <- c(Low = 0, Medium = 1, High = 2, "Very High" = 3)
df_train <- df_train |> mutate(risk_enc = order_map[risk_level])
cat("\nOrdinal encoding sample:\n")
print(df_train |> select(risk_level, risk_enc) |> distinct() |> arrange(risk_enc))

# ── Encoding 3: Frequency encoding ───────────────────────────────────────────
freq_map <- df_train |> count(city) |> mutate(city_freq = n / sum(n)) |> select(city, city_freq)
df_train <- df_train |> left_join(freq_map, by = "city")
cat("\nFrequency encoding (top 5 cities):\n")
print(df_train |> select(city, city_freq) |> distinct() |> arrange(desc(city_freq)) |> head(5))


**Encoding selection guide:**

| Situation | Encoding | Why |
| :--- | :--- | :--- |
| Nominal, low cardinality (<15), linear model | `step_dummy()` | No implied order |
| Ordinal (Low/Med/High) | Ordered factor / integer map | Order is meaningful |
| High cardinality (>50), tree model | Frequency | Avoids sparse columns |
| High cardinality, strong target signal | Target (with CV) | Most predictive — but leakage risk |

**Common mistakes:**
- Label-encoding a nominal column (e.g. `city`) for a linear model — implies `Paris > London > Tokyo` which is meaningless and distorts coefficients
- Forgetting that `step_dummy()` drops one level by default — this IS the `drop_first=True` behavior, so unlike pandas (where forgetting the argument is the bug), R's default is already correct for linear models; the trap runs the *other* direction here — remembering to pass `one_hot = TRUE` for tree models when you specifically don't want a level dropped
- Computing target encoding on the full training set without CV — your model sees the target in its features, causing massive overfitting

**A practical §1 footgun: train/test factor levels can mismatch.** Encoding
the two sets separately yields *different columns* whenever a category is
absent from one — the model then breaks at inference. recipes solves this
structurally: `step_dummy()` is **prepped on train and baked on test**, so the
column set it produces for test is locked to what it saw during `prep()` — a
category unseen in test simply gets a row of all zeros, no realignment step
needed. (Frequency encoding still has the sibling issue: categories unseen in
train map to `NA` via the `left_join` above — add `replace_na()`.)

In [ ]:
library(recipes)

# Demonstrate the train/test vocabulary alignment problem

train <- tibble::tibble(platform = c("ios","android","web"))
test  <- tibble::tibble(platform = c("ios","android"))   # 'web' absent from test

# Wrong way: encode each set separately
train_manual <- model.matrix(~ platform - 1, data = train)
test_manual  <- model.matrix(~ platform - 1, data = test)
cat("Manual encoding -- train columns:", colnames(train_manual), "\n")
cat("Manual encoding -- test  columns:", colnames(test_manual), "\n")
cat("MISMATCH: test is missing 'platformweb' -- model will fail at inference\n\n")

# Correct way: prep() on train, bake() on test
rec <- recipe(~ platform, data = train) |>
  step_dummy(platform, one_hot = TRUE) |>
  prep(training = train)

tr_baked <- bake(rec, new_data = NULL)
te_baked <- bake(rec, new_data = test)
cat("recipes prep/bake -- train columns:", names(tr_baked), "\n")
cat("recipes prep/bake -- test  columns:", names(te_baked), "\n")
cat("'platform_web' exists in test (= 0):", "platform_web" %in% names(te_baked), "\n")
print(te_baked)


---
## §2 — Numerical Transformations

Scaling and transforming numeric features matters for distance-based models
(logistic regression, SVM, KNN, neural nets) but not for tree-based models
(XGBoost, Random Forest). Always `prep()` transforms on train, `bake()` test.

In [ ]:
library(recipes)
library(ggplot2)
library(patchwork)

# Use the df_train defined in cell 3
# (all transforms fit on train, applied to test)
rec_num <- recipe(~ spend, data = df_train) |>
  step_normalize(spend) |>       # Z-score: mean=0, sd=1
  prep(training = df_train)

train_scaled <- bake(rec_num, new_data = NULL)
test_scaled  <- bake(rec_num, new_data = df_test)

cat("Original spend: mean=", round(mean(df_train$spend),2),
    " sd=", round(sd(df_train$spend),2), "\n")
cat("Scaled   spend: mean=", round(mean(train_scaled$spend),4),
    " sd=", round(sd(train_scaled$spend),4), "\n")

# Yeo-Johnson for skewed data
rec_yj <- recipe(~ spend, data = df_train) |>
  step_YeoJohnson(spend) |>
  prep(training = df_train)
yj_train <- bake(rec_yj, new_data = NULL)

p1 <- ggplot(df_train, aes(x=spend)) +
  geom_histogram(bins=30, fill="#3498DB", color="white", alpha=0.85) +
  labs(title="Raw spend", x="spend") + theme_minimal(base_size=10)

p2 <- ggplot(train_scaled, aes(x=spend)) +
  geom_histogram(bins=30, fill="#2ECC71", color="white", alpha=0.85) +
  labs(title="After step_normalize()", x="spend (z-score)") + theme_minimal(base_size=10)

p3 <- ggplot(yj_train, aes(x=spend)) +
  geom_histogram(bins=30, fill="#E74C3C", color="white", alpha=0.85) +
  labs(title="After step_YeoJohnson()", x="spend (transformed)") + theme_minimal(base_size=10)

p1 + p2 + p3 +
  plot_annotation(title="Numerical Transformations",
                  theme=theme(plot.title=element_text(face="bold")))


**Transformation selection guide:**

| Situation | Transformation |
| :--- | :--- |
| Right-skewed (revenue, counts) | `log1p()` |
| Linear/logistic regression, SVM, KNN | `step_normalize()` |
| Neural network inputs | `step_range()` or `step_normalize()` |
| Data has significant outliers | manual median/IQR scaling (no single built-in step) |
| Want to auto-normalize any distribution | `step_YeoJohnson()` |
| Tree-based model (XGBoost, RF) | None required |

**Common mistakes:**
- Fitting the recipe on the full dataset before splitting — leaks test set distribution into the scaler; always `prep(training = X_train)`, never on the combined data
- Scaling the target variable when it isn't needed — scaling `y` for regression is usually unnecessary and complicates interpretation
- Using `log()` on columns with zeros — produces `-Inf`; always use `log1p()` for safety, same trap as NumPy's `np.log()`
- Calling `bake()` before `prep()`, or re-`prep()`-ing on test data by mistake — `prep()` is the ONLY place fitting happens; every subsequent `bake()` call, on train or test, reuses those fitted parameters

---
## §3 — Binning & Discretization

Convert a continuous variable into labeled groups. Useful when the
relationship with the target is non-linear and step-like, or when business
rules define meaningful ranges.

In [ ]:
library(dplyr)

# Use df_train from the SS0 encoding cell
df_binned <- df_train |> mutate(
  spend_group = cut(
    spend,
    breaks = c(0, 100, 250, 500),
    labels = c("Low", "Medium", "High"),
    right = TRUE, include.lowest = TRUE
  )
)
cat("Spend bins:\n"); print(table(df_binned$spend_group))

# Equal-frequency bins (compute on train only)
train_breaks <- quantile(df_train$spend, probs = seq(0,1,0.25), na.rm=TRUE)
df_binned <- df_binned |> mutate(
  spend_quartile = cut(spend, breaks=train_breaks,
                       labels=c("Q1","Q2","Q3","Q4"), include.lowest=TRUE)
)
cat("\nQuartile bins:\n"); print(table(df_binned$spend_quartile))

# Multi-condition segmentation with case_when
df_binned <- df_binned |> mutate(
  user_segment = case_when(
    spend >= 250 & churn == 0 ~ "VIP",
    spend >= 100              ~ "Regular",
    .default                  = "Low Value"
  )
)
cat("\nUser segments:\n"); print(table(df_binned$user_segment))


**`cut()` with fixed breaks vs quantile breaks:**

| | Fixed `breaks=` | `quantile()`-derived `breaks=` |
| :--- | :--- | :--- |
| Bin width | Equal width | Equal frequency |
| Group sizes | Unequal (depends on distribution) | Roughly equal |
| Edges from | You specify | Computed from data |
| Use when | Domain-defined thresholds | Want balanced groups |
| Leakage risk | Low | High — compute on train only |

**Common mistakes:**
- Computing quantile breakpoints on the full dataset — bin edges depend on the distribution, which leaks test set info into the model, identical trap to pandas' `qcut`
- Forgetting `include.lowest = TRUE` — the lowest value falls outside all bins and becomes `NA`
- Using `cut()` labels as a plain factor directly in a linear model without an explicit encoding step — `lm()`/`glm()` actually handle unordered factors correctly via internal dummy coding, but for `recipes`-based workflows, run `step_dummy()` on the binned column afterward to be explicit about the encoding

---
## §4 — Date & Time Features

Raw timestamps are not useful to most models. Extract meaningful components —
and encode cyclical features correctly so models understand that hour 23 is
close to hour 0.

In [ ]:
library(lubridate)
library(dplyr)

# Self-contained datetime sample
set.seed(42)
df_dt <- tibble::tibble(
  user_id  = 1:200,
  ts       = ymd_hms("2024-01-01 00:00:00") +
             seconds(sample(0:(365*24*3600), 200, replace=FALSE)),
  signup   = ymd("2023-01-01") + days(sample(0:365, 200, replace=TRUE))
)

df_dt <- df_dt |> mutate(
  year         = year(ts),
  month        = month(ts),
  day_of_week  = wday(ts, week_start = 1) - 1,   # 0=Mon, 6=Sun
  hour         = hour(ts),
  is_weekend   = as.integer(day_of_week >= 5),
  days_since_signup = as.numeric(as_date(ts) - signup, units = "days"),
  # Cyclical encoding for hour
  hour_sin = sin(2 * pi * hour / 24),
  hour_cos = cos(2 * pi * hour / 24)
)

cat("Date feature sample:\n")
print(df_dt |> select(ts, year, month, day_of_week, hour, is_weekend,
                       days_since_signup, hour_sin, hour_cos) |> head(5))
cat("\nWeekend distribution:\n"); print(table(df_dt$is_weekend))


**When to use cyclical encoding:**

| Feature | Range | Use cyclical? |
| :--- | :--- | :--- |
| Hour of day | 0-23 | Yes — 23 is close to 0 |
| Day of week | 0-6 | Yes — Sunday is close to Monday |
| Month | 1-12 | Yes — December is close to January |
| Year | 2020-2024 | No — not cyclical |
| Days since event | 0-infinity | No — not cyclical |

**Common mistakes:**
- Using raw hour as a numeric feature — model sees hour 1 and hour 23 as far apart; use sin/cos encoding instead
- Computing `days_since_signup`-style lag features without `arrange()`-ing by user and date first — `lag()` operates on row order, not time order
- Extracting too many date features — correlated features (month + quarter + season) add noise; pick the most granular
- Reaching for a US-Python-package-specific holiday calendar name out of habit — `timeDate::holidayNYSE()` (or the broader `holidayNERC()`/custom calendars) is R's standard source for US business-day calendars; there is no exact 1:1 package name match, so verify the specific holiday set matches your business need

---
## §5 — Lag & Window Features

Use past values to predict future values. The most powerful feature type for
time-series and sequential data — and the easiest place to introduce
lookahead leakage.

In [ ]:
library(dplyr)
library(slider)

# Self-contained time-series sample
set.seed(42)
df_ts <- tibble::tibble(
  user_id = rep(1:5, each=12),
  date    = rep(seq(as.Date("2024-01-01"), by="month", length.out=12), 5),
  revenue = round(runif(60, 50, 300), 2)
)

# Lag features (always sort first)
df_ts <- df_ts |> arrange(user_id, date) |> group_by(user_id) |> mutate(
  rev_lag1     = lag(revenue, 1),
  rev_lag3     = lag(revenue, 3),
  rev_delta    = revenue - rev_lag1,
  rev_pct_chg  = rev_delta / na_if(rev_lag1, 0),
  # Rolling mean of PREVIOUS 3 months (lag first to avoid leakage)
  rev_roll3    = slide_dbl(lag(revenue, 1), mean, .before=2, na.rm=TRUE, .complete=FALSE)
) |> ungroup()

cat("Lag and rolling features (user 1):\n")
print(df_ts |> filter(user_id == 1) |>
        select(date, revenue, rev_lag1, rev_delta, rev_roll3) |>
        mutate(across(where(is.numeric), ~round(.x, 2))))

# Train/test split for time series (never random)
cutoff <- as.Date("2024-09-01")
train  <- df_ts |> filter(date <  cutoff)
test   <- df_ts |> filter(date >= cutoff)
cat(sprintf("\nTime split at %s: train=%d rows, test=%d rows\n",
    cutoff, nrow(train), nrow(test)))


**Leakage anatomy for lag features:**

```
Timeline: ... | day 5 | day 6 | day 7 (target) |

lag1 = day 6 revenue                            <- OK: past data
lag7 = day 0 revenue                             <- OK: past data
rolling_mean WITHOUT lag(1) wrap = mean(day 5, 6, 7)  <- LEAKAGE: includes target day
rolling_mean WITH lag(1) wrap    = mean(day 4, 5, 6)  <- OK: excludes target day
```

**Common mistakes:**
- Not wrapping the series in `lag(x, 1)` before `slide_dbl()` — the rolling window includes the current row, which is the target, the exact R analog to forgetting `.shift(1)` before `.rolling()`
- Using a random `initial_split()` for time series — future rows leak into training via lag features; use `initial_time_split()` or a manual date cutoff instead
- Not `arrange()`-ing by date within each group before `lag()` — shift operates on row position, not time order
- Forgetting `.complete = FALSE` in `slide_dbl()` — without it, the first N-1 rows of each group become `NA` *and stay that way regardless of available data*; `.complete = FALSE` is `slider`'s analog to pandas' `min_periods=1`, computing a shorter window rather than discarding the row

---
## §6 — Interaction & Ratio Features

Combining two or more columns can capture relationships that neither column
captures alone. Most valuable when domain knowledge suggests the combination
is meaningful.

In [ ]:
library(dplyr)
library(recipes)

# Use df_train from cell 3 (has spend and churn)
df_feat <- df_train |> mutate(
  # Ratio features
  spend_per_visit = spend / pmax(1, round(spend / 50)),   # illustrative
  # Group-relative features
  group_mean = mean(spend, na.rm=TRUE),
  spend_vs_mean = spend - group_mean,
  # Interaction via recipes
  spend_x_churn = spend * churn
)

cat("Ratio and interaction features:\n")
print(df_feat |> select(spend, spend_per_visit, spend_vs_mean, spend_x_churn) |> head(6))

# step_poly + step_interact via recipes
rec_int <- recipe(~ spend + risk_enc, data = df_train) |>
  step_poly(spend, degree = 2) |>
  step_interact(terms = ~ spend_poly_1:risk_enc) |>
  prep(training = df_train)
int_out <- bake(rec_int, new_data = NULL)
cat("\nPolynomial + interaction features:\n")
print(names(int_out))


**Common mistakes:**
- Dividing without handling zeros — `clicks / impressions` when `impressions = 0` produces `Inf` or `NaN`; always `na_if(denominator, 0)` first, R's direct analog to pandas' `.replace(0, np.nan)`
- Creating all possible interactions automatically without domain guidance — most interactions are noise; domain-motivated features outperform automated `step_poly()`/`step_interact()` features
- Not checking for multicollinearity after adding ratio features — `rev_per_user`, `total_rev`, and `user_count` are linearly dependent; use `car::vif()` or a correlation matrix (`cor()`) to diagnose

---
## §7 — Text as Features

For tabular ML problems with one or two text columns — not full NLP, just
enough to extract signal from free text. `textrecipes` extends the
`recipes` framework with text-specific `step_*()` functions, the direct
tidymodels analog to scikit-learn's `feature_extraction.text` module.

In [ ]:
library(stringr)
library(dplyr)
library(textrecipes)
library(recipes)

# Self-contained text sample
df_text <- tibble::tibble(
  id          = 1:100,
  description = sample(c(
    "fast shipping great product love it",
    "poor quality bad packaging disappointed",
    "excellent value amazing quality highly recommend",
    "slow delivery item damaged on arrival",
    "good product fair price would buy again"
  ), 100, replace=TRUE),
  label = rbinom(100, 1, 0.5)
)
train_text <- df_text[1:70, ]; test_text <- df_text[71:100, ]

# Simple text features (no vectorization needed)
train_text <- train_text |> mutate(
  word_count   = str_count(description, "\\S+"),
  has_positive = as.integer(str_detect(description, "great|excellent|amazing|good")),
  has_negative = as.integer(str_detect(description, "poor|bad|disappointed|damaged"))
)
cat("Simple text features:\n")
print(train_text |> select(description, word_count, has_positive, has_negative) |> head(4))

# TF-IDF via textrecipes
rec_tfidf <- recipe(~ description, data = train_text) |>
  step_tokenize(description) |>
  step_stopwords(description) |>
  step_tokenfilter(description, max_tokens = 20) |>
  step_tfidf(description) |>
  prep(training = train_text)

tfidf_train <- bake(rec_tfidf, new_data = NULL)
tfidf_test  <- bake(rec_tfidf, new_data = test_text)
cat(sprintf("\nTF-IDF features: %d columns\n", ncol(tfidf_train)))
cat("Feature names:", paste(names(tfidf_train)[1:5], collapse=", "), "...\n")


**Common mistakes:**
- `prep()`-ing the text recipe on the full dataset — vocabulary and IDF weights leak test distribution; always `prep(training = X_train)`
- Not filling `NA` before tokenizing — `step_tokenize()` errors on `NA` values; use `mutate(description = replace_na(description, ""))` first, identical trap to pandas' `TfidfVectorizer` on un-filled NaN
- Using too many `max_tokens` — 100-500 is usually enough; more features add noise and memory pressure

---
## §8 — Leakage & Validation

Feature leakage is when information from the future (or from the target
itself) is encoded into features used to train the model. It produces
unrealistically high training performance and catastrophic failure in
production.

In [ ]:
library(dplyr)
library(rsample)
library(recipes)
library(lubridate)

# Sample data for leakage examples
set.seed(42)
n <- 500
df <- tibble::tibble(
  user_id  = rep(1:50, each = 10),
  date     = rep(seq(as.Date("2024-01-01"), by = "month", length.out = 10), 50),
  revenue  = rnorm(n, 100, 30),
  feature1 = rnorm(n),
  feature2 = rnorm(n)
)
y_target  <- rbinom(n, 1, 0.3)
X_train   <- df |> select(feature1, feature2, revenue) |> slice(1:350)
X_test    <- df |> select(feature1, feature2, revenue) |> slice(351:n)
y_train   <- y_target[1:350]

# ── Type 1: Target leakage -- feature is computed using the target ─────────────
# Example: predicting loan default, and a feature is 'days_past_due'
# days_past_due is 0 for non-defaulters and >0 for defaulters -- it IS the target

# Detection: check correlation of each feature with the target
correlations <- sapply(X_train, function(col) if (is.numeric(col)) abs(cor(col, y_train, use = "complete.obs")) else NA)
cat("Feature correlations with target:\n")
print(sort(correlations, decreasing = TRUE, na.last = NA))
cat("Suspicious: correlation > 0.8 -> investigate that feature\n\n")

# ── Type 2: Temporal leakage -- using future data for a past prediction ────────
df_sorted     <- df |> arrange(date)
split_correct <- initial_time_split(df_sorted, prop = 0.8)
train         <- training(split_correct)
test          <- testing(split_correct)
cutoff        <- quantile(df$date, 0.8)
cat(sprintf("Time-based split cutoff: %s\n", as.character(as.Date(cutoff, origin = "1970-01-01"))))
cat(sprintf("Train: %d rows, Test: %d rows\n", nrow(train), nrow(test)))

# ── Type 3: Group leakage -- same entity in train and test ─────────────────────
split_group  <- group_initial_split(df, group = user_id, prop = 0.8)
X_train_grp  <- training(split_group)
X_test_grp   <- testing(split_group)
cat(sprintf("Group split -- users in train: %d, in test: %d, overlap: %d\n",
    n_distinct(X_train_grp$user_id), n_distinct(X_test_grp$user_id),
    length(intersect(unique(X_train_grp$user_id), unique(X_test_grp$user_id)))))

# ── Safe preprocessing pipeline -- prep on train, bake both ───────────────────
rec <- recipe(~ feature1 + feature2 + revenue, data = X_train) |>
  step_impute_median(all_numeric_predictors()) |>
  step_normalize(all_numeric_predictors()) |>
  prep(training = X_train)

X_train_processed <- bake(rec, new_data = NULL)
X_test_processed  <- bake(rec, new_data = X_test)
cat(sprintf("\nProcessed train shape: %d x %d\n", nrow(X_train_processed), ncol(X_train_processed)))
cat("prep() on train only -- bake() applies the same transform to test (no re-fitting)\n")


**Leakage types and fixes:**

| Type | Signal | Fix |
| :--- | :--- | :--- |
| Target leakage | Feature correlates >0.8 with target; comes from post-event data | Remove the feature; ask "would I know this at prediction time?" |
| Temporal leakage | Random split on time-series; rolling features without `lag(1)` | `initial_time_split()` / cutoff filter; always wrap in `lag(1)` before `slide_dbl()` |
| Group leakage | Same entity ID in train and test | `group_initial_split()` or entity-level split |
| Preprocessing leakage | Recipe `prep()`-ed on full data before split | `prep(training = X_train)` only, never on combined data |
| Target encoding leakage | Category means computed without CV | Use out-of-fold CV target encoding (see §1) |

**Common mistakes:**
- Treating suspiciously high accuracy as a win — if a churn model hits 99% AUC, check for leakage before celebrating
- Filling missing values before splitting — the imputed value (e.g. median) is computed from the full dataset, leaking test statistics into train; this is exactly what `step_impute_median()` inside a `prep(training = X_train)`-scoped recipe prevents structurally
- Manually computing scaling/imputation statistics outside a `recipe()` and applying them by hand to both sets — the `recipe()`/`prep()`/`bake()` pattern enforces correct fit-on-train behavior the same way `sklearn.Pipeline` does; bypassing it reintroduces the exact risk it exists to close off

---
## Decision Guide

```
Encoding a categorical column?
+-- Nominal, <15 unique values, linear model    -> step_dummy() (one level dropped by default)  (§1)
+-- Ordinal (Low/Med/High)                      -> ordered factor + as.integer()                (§1)
+-- High cardinality, tree model                -> frequency encoding (count() + left_join)      (§1)
\-- High cardinality, strong target signal      -> target encoding with CV (vfold_cv)            (§1)

Transforming a numeric column?
+-- Right-skewed (revenue, counts)              -> log1p()                                       (§2)
+-- Linear model, SVM, KNN, NN                  -> step_normalize() (fit on train only)          (§2)
+-- Has significant outliers                    -> manual median/IQR scaling                     (§2)
\-- Want auto-normalization                     -> step_YeoJohnson()                              (§2)

Binning a continuous variable?
+-- Business-defined thresholds                 -> cut(breaks = ...)                              (§3)
+-- Equal-sized groups                          -> quantile() breaks on train only, then cut()    (§3)
\-- Multi-condition business logic              -> case_when()                                    (§3)

Extracting from a datetime column?
+-- Cyclical component (hour, month, weekday)   -> sin/cos encoding                              (§4)
+-- Time elapsed since an event                 -> as.numeric(as_date(ts) - ref_date)             (§4)
\-- Time since last event per user              -> group_by() |> lag() -> difftime in hours       (§4)

Creating lag/window features?
+-- Previous value                              -> group_by() |> lag()                            (§5)
+-- Rolling mean/std over last N rows           -> slide_dbl(lag(x,1), fn, .before=N-1)           (§5)
+-- Cumulative sum from start of series         -> slide_dbl(lag(x,1), sum, .before=Inf)          (§5)
\-- Train/test split                            -> initial_time_split() / date cutoff, NOT random (§5)

Combining two columns?
+-- One normalizes the other (CTR, ARPU)        -> ratio feature (na_if() the denominator)       (§6)
+-- Gap from a reference (vs group mean)        -> difference feature                             (§6)
\-- Both features interact                      -> product feature / step_interact()              (§6)

Text column in a tabular dataset?
+-- Simple signal (length, keywords, counts)    -> str_length(), str_detect()                    (§7)
\-- Richer vocabulary signal                    -> textrecipes::step_tfidf() (fit on train only)  (§7)

Suspecting leakage?
+-- Model accuracy is unrealistically high      -> check feature correlations with target         (§8)
+-- Time-series with random split               -> switch to initial_time_split()                 (§8)
+-- Same user in train and test                 -> group_initial_split()                          (§8)
\-- Preprocessing fit on full data              -> use a recipe(), prep(training=X_train) only    (§8)
```

**§8 capstone: the full `recipe()`, the direct structural analog to
`ColumnTransformer`.** §8 shows individual steps in isolation, but real
feature sets mix column types in one object. Unlike scikit-learn (where
`ColumnTransformer` is a *separate* wrapper needed specifically because
individual transformers are column-blind), `recipes::recipe()` **already**
routes each `step_*()` to the right columns via `tidyselect` — there is no
extra wrapping layer to learn. `prep()` fits on train; `bake()` applies
to test.

In [ ]:
library(recipes)

X_train <- tibble::tibble(
  age      = c(25, 40, NA, 33),
  platform = c('ios','web','ios','android')
)

rec <- recipe(~ age + platform, data = X_train) |>
  step_impute_median(age) |>                              # numeric pipeline, step 1: impute
  step_normalize(age) |>                                   # numeric pipeline, step 2: scale
  step_dummy(platform, one_hot = TRUE) |>                  # categorical pipeline: one-hot encode
  prep(training = X_train)                                  # fit on TRAIN only; use bake(rec, new_data = X_test) on test

out <- bake(rec, new_data = NULL)
cat('output shape :', paste(dim(out), collapse = " x "), '\n')
cat('feature names:', paste(names(out), collapse = ", "), '\n')